# SQL Analysis

This notebook uses SQL to analyze cleaned Divvy bike-sharing trip data for January–July 2026.
The analysis focuses on:
- overall usage trends
- member vs casual rider behavior
- monthly and weekly patterns
- peak usage hours
- ride duration
- bike type preferences
- popular stations and routes

In [ ]:
%pip install duckdb

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import duckdb
import pandas as pd

print(duckdb.__version__)

1.5.6


In [ ]:
con = duckdb.connect()

In [ ]:
csv_path = "processed/divvy_trips_cleaned_2026_01_07.csv"

con.execute(f"""
CREATE OR REPLACE VIEW rides AS
SELECT *
FROM read_csv_auto('{csv_path}')
""")

In [ ]:
con.execute(f"""
CREATE OR REPLACE VIEW rides AS
SELECT *
FROM read_csv_auto('{csv_path}')
""")

In [ ]:
con.execute("""
SELECT COUNT(*) AS total_rides
FROM rides
""").df()

,total_rides
0,3297649


In [ ]:
con.execute("""
SELECT
    COUNT(*) AS total_rides,
    COUNT(DISTINCT ride_id) AS unique_rides,
    ROUND(AVG(ride_duration_min), 2) AS avg_ride_duration_min,
    ROUND(MEDIAN(ride_duration_min), 2) AS median_ride_duration_min,
    MIN(started_at) AS first_trip,
    MAX(started_at) AS last_trip
FROM rides
""").df()

,total_rides,unique_rides,avg_ride_duration_min,median_ride_duration_min,first_trip,last_trip
0,3297649,3297649,14.41,9.51,2026-01-01 00:00:50.683,2026-07-31 23:57:02.949


In [ ]:
con.execute("""
SELECT
    member_casual,
    COUNT(*) AS total_rides,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS share_percent,
    ROUND(AVG(ride_duration_min), 2) AS avg_ride_duration_min,
    ROUND(MEDIAN(ride_duration_min), 2) AS median_ride_duration_min
FROM rides
GROUP BY member_casual
ORDER BY total_rides DESC
""").df()

,member_casual,total_rides,share_percent,avg_ride_duration_min,median_ride_duration_min
0,member,2143281,64.99,12.15,8.68
1,casual,1154368,35.01,18.59,11.37


In [ ]:
monthly_usage = con.execute("""
SELECT
    month,
    month_name,
    COUNT(*) AS total_rides,
    ROUND(AVG(ride_duration_min), 2) AS avg_ride_duration_min
FROM rides
GROUP BY month, month_name
ORDER BY month
""").df()

monthly_usage

,month,month_name,total_rides,avg_ride_duration_min
0,1,January,133743,11.86
1,2,February,197317,11.45
2,3,March,308466,12.87
3,4,April,435478,13.05
4,5,May,635395,15.15
5,6,June,741382,15.50
6,7,July,845868,15.24


In [ ]:
con.execute("""
SELECT
    month,
    month_name,
    member_casual,
    COUNT(*) AS total_rides
FROM rides
GROUP BY
    month,
    month_name,
    member_casual
ORDER BY
    month,
    member_casual
""").df()

,month,month_name,member_casual,total_rides
0,1,January,casual,23878
1,1,January,member,109865
2,2,February,casual,40077
3,2,February,member,157240
4,3,March,casual,84812
5,3,March,member,223654
6,4,April,casual,127018
7,4,April,member,308460
8,5,May,casual,235873
9,5,May,member,399522


### Key Findings

- The cleaned dataset contains 3,297,649 unique trips between January and July 2026.
- Members account for approximately 65% of all rides, while casual users represent about 35%.
- Casual riders take longer trips than members: the average ride duration is 18.59 minutes for casual users compared with 12.15 minutes for members.
- Overall bike usage increases substantially across the analyzed period, from 133,743 rides in January to 845,868 rides in July.
- Average ride duration is also higher during the later months of the analyzed period, reaching approximately 15 minutes between May and July.
- Both member and casual usage increase over time, with casual ridership becoming considerably more prominent during the later months.

In [ ]:
weekly_usage = con.execute("""
SELECT
    day_of_week_num,
    day_of_week,
    member_casual,
    COUNT(*) AS total_rides,
    ROUND(AVG(ride_duration_min), 2) AS avg_ride_duration_min
FROM rides
GROUP BY
    day_of_week_num,
    day_of_week,
    member_casual
ORDER BY
    day_of_week_num,
    member_casual
""").df()

weekly_usage

,day_of_week_num,day_of_week,member_casual,total_rides,avg_ride_duration_min
0,0,Monday,casual,133641,18.80
1,0,Monday,member,300749,11.94
2,1,Tuesday,casual,134985,16.11
3,1,Tuesday,member,348950,11.67
4,2,Wednesday,casual,135738,15.64
5,2,Wednesday,member,344976,11.63
6,3,Thursday,casual,144915,16.17
7,3,Thursday,member,342816,11.68
8,4,Friday,casual,179080,18.13
9,4,Friday,member,312581,12.10


In [ ]:
con.execute("""
SELECT
    is_weekend,
    member_casual,
    COUNT(*) AS total_rides,
    ROUND(AVG(ride_duration_min), 2) AS avg_ride_duration_min
FROM rides
GROUP BY
    is_weekend,
    member_casual
ORDER BY
    is_weekend,
    member_casual
""").df()

,is_weekend,member_casual,total_rides,avg_ride_duration_min
0,False,casual,728359,17.02
1,False,member,1650072,11.80
2,True,casual,426009,21.28
3,True,member,493209,13.34


In [ ]:
hourly_usage = con.execute("""
SELECT
    start_hour,
    member_casual,
    COUNT(*) AS total_rides
FROM rides
GROUP BY
    start_hour,
    member_casual
ORDER BY
    start_hour,
    member_casual
""").df()

hourly_usage

,start_hour,member_casual,total_rides
0,0,casual,24156
1,0,member,19298
2,1,casual,15599
3,1,member,11710
4,2,casual,9625
5,2,member,6710
6,3,casual,5678
7,3,member,4616
8,4,casual,4492
9,4,member,5412


In [ ]:
con.execute("""
SELECT
    start_hour,
    COUNT(*) AS total_rides
FROM rides
GROUP BY start_hour
ORDER BY total_rides DESC
LIMIT 10
""").df()

,start_hour,total_rides
0,17,343944
1,16,304161
2,18,279170
3,15,225308
4,19,201281
5,8,197642
6,14,187013
7,13,184455
8,12,180601
9,11,154021


### Usage Pattern Findings

- Member riders show stronger weekday usage, while casual ridership increases substantially toward the weekend.
- Saturday is the busiest day for casual riders, with 245,246 trips, whereas member usage is highest during the middle of the working week.
- Approximately 36.9% of casual rides occur on weekends, compared with about 23.0% of member rides.
- Weekend rides are longer for both rider groups. Casual riders average 21.28 minutes per weekend ride compared with 17.02 minutes on weekdays, while members average 13.34 minutes compared with 11.80 minutes.
- Overall demand peaks at 17:00, followed by 16:00 and 18:00.
- Member usage shows distinct morning and evening peaks, particularly around 7–8 AM and 4–6 PM. This pattern is consistent with regular commuting or work-related travel.
- Casual usage is less concentrated in the morning and rises more gradually throughout the day, suggesting different usage behaviour from members.

In [ ]:
bike_type_usage = con.execute("""
SELECT
    rideable_type,
    member_casual,
    COUNT(*) AS total_rides,
    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (PARTITION BY member_casual),
        2
    ) AS share_within_user_type,
    ROUND(AVG(ride_duration_min), 2) AS avg_ride_duration_min
FROM rides
GROUP BY
    rideable_type,
    member_casual
ORDER BY
    member_casual,
    total_rides DESC
""").df()

bike_type_usage

,rideable_type,member_casual,total_rides,share_within_user_type,avg_ride_duration_min
0,electric_bike,casual,844838,73.19,14.74
1,classic_bike,casual,309530,26.81,29.11
2,electric_bike,member,1455117,67.89,11.35
3,classic_bike,member,688164,32.11,13.85


In [ ]:
top_start_stations = con.execute("""
SELECT
    start_station_name,
    COUNT(*) AS total_rides
FROM rides
WHERE start_station_name IS NOT NULL
GROUP BY start_station_name
ORDER BY total_rides DESC
LIMIT 15
""").df()
top_start_stations

,start_station_name,total_rides
0,Navy Pier,38131
1,DuSable Lake Shore Dr & Monroe St,22072
2,Michigan Ave & Oak St,19886
3,DuSable Lake Shore Dr & North Blvd,19258
4,State St & Chicago Ave,16738
5,Canal St & Madison St,16135
6,Kingsbury St & Kinzie St 2,15752
7,Wells St & Concord Ln,15607
8,Theater on the Lake,15538
9,Wells St & Elm St,15251


In [ ]:
con.execute("""
WITH ranked_stations AS (
    SELECT
        member_casual,
        start_station_name,
        COUNT(*) AS total_rides,
        ROW_NUMBER() OVER (
            PARTITION BY member_casual
            ORDER BY COUNT(*) DESC
        ) AS station_rank
    FROM rides
    WHERE start_station_name IS NOT NULL
    GROUP BY
        member_casual,
        start_station_name
)
SELECT
    member_casual,
    start_station_name,
    total_rides,
    station_rank
FROM ranked_stations
WHERE station_rank <= 10
ORDER BY
    member_casual,
    station_rank
""").df()

,member_casual,start_station_name,total_rides,station_rank
0,casual,Navy Pier,28958,1
1,casual,DuSable Lake Shore Dr & Monroe St,17253,2
2,casual,Michigan Ave & Oak St,12111,3
3,casual,DuSable Lake Shore Dr & North Blvd,10759,4
4,casual,Millennium Park,9381,5
5,casual,Shedd Aquarium,9340,6
6,casual,Theater on the Lake,8856,7
7,casual,Dusable Harbor,7781,8
8,casual,Montrose Harbor,6014,9
9,casual,Adler Planetarium,5639,10


In [ ]:
top_routes = con.execute("""
SELECT
    start_station_name,
    end_station_name,
    COUNT(*) AS total_rides
FROM rides
WHERE
    start_station_name IS NOT NULL
    AND end_station_name IS NOT NULL
GROUP BY
    start_station_name,
    end_station_name
ORDER BY total_rides DESC
LIMIT 15
""").df()

top_routes

,start_station_name,end_station_name,total_rides
0,Navy Pier,Navy Pier,6074
1,DuSable Lake Shore Dr & Monroe St,DuSable Lake Shore Dr & Monroe St,3608
2,DuSable Lake Shore Dr & Monroe St,Navy Pier,3023
3,Michigan Ave & Oak St,Michigan Ave & Oak St,2368
4,Ellis Ave & 60th St,Ellis Ave & 55th St,1880
5,Ellis Ave & 55th St,Ellis Ave & 60th St,1732
6,Navy Pier,DuSable Lake Shore Dr & Monroe St,1643
7,Ellis Ave & 60th St,University Ave & 57th St,1549
8,University Ave & 57th St,Ellis Ave & 60th St,1405
9,Navy Pier,Theater on the Lake,1390


In [ ]:
con.execute("""
SELECT
    start_station_name,
    end_station_name,
    COUNT(*) AS total_rides
FROM rides
WHERE
    start_station_name IS NOT NULL
    AND end_station_name IS NOT NULL
    AND start_station_name <> end_station_name
GROUP BY
    start_station_name,
    end_station_name
ORDER BY total_rides DESC
LIMIT 15
""").df()

,start_station_name,end_station_name,total_rides
0,DuSable Lake Shore Dr & Monroe St,Navy Pier,3023
1,Ellis Ave & 60th St,Ellis Ave & 55th St,1880
2,Ellis Ave & 55th St,Ellis Ave & 60th St,1732
3,Navy Pier,DuSable Lake Shore Dr & Monroe St,1643
4,Ellis Ave & 60th St,University Ave & 57th St,1549
5,University Ave & 57th St,Ellis Ave & 60th St,1405
6,Navy Pier,Theater on the Lake,1390
7,Navy Pier,DuSable Lake Shore Dr & North Blvd,1284
8,Shedd Aquarium,Navy Pier,1221
9,Navy Pier,Millennium Park,1193


### Bike Type and Station Findings

- Electric bikes are the dominant bike type for both rider groups, accounting for 73.19% of casual rides and 67.89% of member rides.
- Casual riders use classic bikes for substantially longer trips, averaging 29.11 minutes compared with 14.74 minutes for electric-bike trips.
- For members, the difference in ride duration between classic and electric bikes is much smaller: 13.85 versus 11.35 minutes.
- Rider groups show clearly different station usage patterns. Popular casual-rider stations include Navy Pier, Millennium Park, Shedd Aquarium, and Adler Planetarium, while member activity is more concentrated around central transport and business-area stations.
- These station patterns are consistent with casual riders using the service relatively more for leisure and recreational travel, while members appear to use it more frequently for regular urban mobility.
- Several of the most common routes begin and end at the same station. Navy Pier → Navy Pier is the most frequent such route, with 6,074 rides, suggesting a meaningful amount of round-trip recreational usage.
- The most frequent route between different stations is DuSable Lake Shore Dr & Monroe St → Navy Pier, with 3,023 rides.

In [ ]:
con.execute("""
SELECT
    member_casual,
    CASE
        WHEN start_station_name = end_station_name THEN 'Round trip'
        ELSE 'Different station'
    END AS trip_pattern,
    COUNT(*) AS total_rides,
    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (PARTITION BY member_casual),
        2
    ) AS share_percent,
    ROUND(AVG(ride_duration_min), 2) AS avg_ride_duration_min
FROM rides
WHERE
    start_station_name IS NOT NULL
    AND end_station_name IS NOT NULL
GROUP BY
    member_casual,
    trip_pattern
ORDER BY
    member_casual,
    total_rides DESC
""").df()

,member_casual,trip_pattern,total_rides,share_percent,avg_ride_duration_min
0,casual,Different station,709339,92.11,19.57
1,casual,Round trip,60749,7.89,34.54
2,member,Different station,1449734,97.79,12.10
3,member,Round trip,32837,2.21,22.80


### Round-Trip Findings

- Round trips are considerably more common among casual riders: 7.89% of casual trips start and end at the same station, compared with only 2.21% of member trips.
- Round trips are also substantially longer than trips between different stations.
- Casual round trips average 34.54 minutes, compared with 19.57 minutes for trips ending at a different station.
- Member round trips average 22.80 minutes, compared with 12.10 minutes for trips between different stations.
- These patterns further suggest that casual riders use the service relatively more for recreational or leisure-oriented trips, while members more often use bikes for point-to-point transportation.